# Week3_ex1 - Multi-Turn Coil Around a Straight Wire (Magnetic Flux Linkage)

This one builds a multi-turn ring coil around a straight current-carrying wire and checks the flux linkage through the coil using both a winding excitation and a data table report.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil

In [ ]:
# 1. AEDT desktop interface
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. turn off autosave
DT.disable_autosave()

# 3. create project and set solution type
sol_type = "Magnetostatic"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. odesign object for script-recording style API calls
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##
proj_name = "Week3_ex1"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

desi_name = "Week3_ex1"

## Save project and apply design name ##
proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()

In [ ]:
# Helper function to find the terminal faces of the straight wire
# Looks at the face centers of the object, picks the face with the largest |x|,
# then returns both terminal faces sorted by x position (outer, inner)
def find_terminal_face(winding_obj):
    terminal_face = []

    # find maximum x position among the object's faces
    max_x_pos = max(abs(face.center[0]) for face in winding_obj.faces)

    # collect the faces sitting at that max x position (the two terminals)
    for face in winding_obj.faces:
        if abs(abs(face.center[0]) - max_x_pos) <= 0.0001:
            terminal_face.append(face)

    # sort by x so we know which one is "outer" vs "inner"
    ter_out, ter_in = sorted(terminal_face, key=lambda x: x.center[0], reverse=True)

    return ter_out, ter_in

In [ ]:
## Design variables ##
line_length = 400
M3D["line_length"] = f"{line_length}mm"

line_diameter = 10
M3D["line_diameter"] = f"{line_diameter}mm"

line_num_seg = 36
M3D["line_num_seg"] = f"{line_num_seg}"

Current = 100
M3D["Current"] = f"{Current}A"

turns = 5
M3D["turns"] = f"{turns}"

gap = 1.5
M3D["gap"] = f"{gap}mm"

In [ ]:
## Create and configure analysis setup ##

my_setup = M3D.create_setup(name="Setup1")

my_setup.props["MaximumPasses"] = 20
my_setup.props["PercentRefinement"] = 30
my_setup.props["PercentError"] = 1
my_setup.props["RelativeResidual"] = 1e-06
my_setup.props["NonLinearResidual"] = 0.001
my_setup.props["SmoothBHCurve"] = False
my_setup.update()


In [ ]:
## Draw the straight wire ##
point_tmp = []
point_tmp.append(["-line_length/2", "0mm", "0mm"])
point_tmp.append(["line_length/2", "0mm", "0mm"])
line = M3D.modeler.create_polyline(points=point_tmp, name="line", material="copper",
                            xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)


## Create region ##
region_tmp = ["line_length/2", "-line_length/2", "line_length*3", "-line_length*3", "line_length*3", "-line_length*3"]
region = M3D.modeler.create_region(pad_value=region_tmp, pad_type="Absolute Position")


## Draw ring coil (two cylinders, subtract inner from outer) ##
center = [0, "-line_length*(1/4)", "-0.25mm"]
origin = [0, "-line_length*(1/4)-20.25mm", "-0.25mm"]
ring_2 = M3D.modeler.create_polyhedron(orientation="Z", center=center, origin=origin, height="0.5mm", num_sides=36, name="ring_2", material="copper")

center = [0, "-line_length*(1/4)", "-0.25mm"]
origin = [0, "-line_length*(1/4)-19.75mm", "-0.25mm"]
tmp = M3D.modeler.create_polyhedron(orientation="Z", center=center, origin=origin, height="0.5mm", num_sides=36, name=None, material=None)

M3D.modeler.subtract(blank_list=ring_2, tool_list=tmp, keep_originals=False)


## Duplicate ring coil to build the multi-turn stack ##
vector = [0, 0, "gap"]
M3D.modeler.duplicate_along_line(assignment=ring_2, vector=vector, clones="turns", attach=False, is_3d_comp=False, duplicate_assignment=True)

coils = [ring_2]
for i in range(1, turns):
    coils.append(M3D.modeler.object_list[-turns+i])

vector = [0, 0, "-gap*(turns-1)/2"]
M3D.modeler.move(assignment=coils, vector=vector)

In [ ]:
## Assign the wire's terminal faces as coil ports ##
ter_out, ter_in = find_terminal_face(line)

M3D.assign_coil(assignment=ter_out, conductors_number=1, polarity="Negative", name="Out")
M3D.assign_coil(assignment=ter_in, conductors_number=1, polarity="Positive", name="In")


## Create the winding for the straight wire and add both terminals to it ##
coil_1 = M3D.assign_winding(assignment=None, winding_type='Current', is_solid=True, current="Current",
                   resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0,
                   name="coil_1")

M3D.add_winding_coils(coil_1.name, coils=["Out", "In"])

In [ ]:
## Create terminal faces for the multi-turn coil via section + separate_bodies ##
M3D.modeler.section(assignment=coils, plane="YZ", create_new=True, section_cross_object=False)

coil_sections = []
for i in range(0, turns):
    coil_sections.append(M3D.modeler.object_list[-turns+i])

M3D.modeler.separate_bodies(assignment=coil_sections, create_group=False)


## Pick the correct terminal by comparing y-coordinates ##
coil_sections_tmp = []
for i in range(0, turns):
    coil_sections_tmp.append(M3D.modeler.object_list[-turns+i])

for i in range(len(coil_sections)):
    y_coil_section = coil_sections[i].faces[0].center[1]
    y_coil_section_tmp = coil_sections_tmp[i].faces[0].center[1]

    # replace with the tmp section if it sits at a larger y
    if y_coil_section_tmp > y_coil_section:
        coil_sections[i] = coil_sections_tmp[i]


## Assign each selected terminal as a coil ##
coil_terminals = []
for i in range(len(coil_sections)):
    coil_terminals.append(M3D.assign_coil(assignment=coil_sections[i], conductors_number=1, polarity='Positive', name=None))


## Create the winding for the multi-turn coil and add all terminals to it ##
coil_2 = M3D.assign_winding(assignment=None, winding_type='Voltage', is_solid=False,
                            current=0, resistance="1e-9", inductance=0, voltage=0, parallel_branches=1, phase=0, name="coil_2")

for i in range(len(coil_terminals)):
    M3D.add_winding_coils(assignment=coil_2.name, coils=coil_terminals[i].name)

In [ ]:
## Run analysis ##
my_setup.analyze()

In [ ]:
## Create the flux linkage vs. line_length data table report (script recording) ##
oModule = oDesign.GetModule("ReportSetup")

oModule.CreateReport("Winding Table 1", "Magnetostatic", "Data Table", "Setup1 : LastAdaptive", [],
	[
		"line_length:=", ["All"],
		"line_diameter:=", ["Nominal"],
		"line_num_seg:=", ["Nominal"],
		"Current:=", ["Nominal"],
		"turns:=", ["Nominal"],
		"gap:=", ["Nominal"]
	],
	[
		"X Component:=", "line_length",
		"Y Component:=", ["FluxLinkage(coil_2)"]
	])

oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers",
				"Winding Table 1:FluxLinkage(coil_2):Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Field Precision",
					"Value:=", "10"
				]
			]
		]
	])

In [ ]:
## Create the B-field overlay on the YZ cut plane (script recording) ##
oModule = oDesign.GetModule("FieldsReporter")

oModule.CreateFieldPlot(
	[
		"NAME:Mag_B1",
		"SolutionName:=", "Setup1 : LastAdaptive",
		"UserSpecifyName:=", 0,
		"UserSpecifyFolder:=", 0,
		"QuantityName:=", "Mag_B",
		"PlotFolder:=", "B",
		"StreamlinePlot:=", False,
		"AdjacentSidePlot:=", False,
		"FullModelPlot:=", False,
		"IntrinsicVar:=", "",
		"PlotGeomInfo:=", [1,"Surface","CutPlane",1,"Global:YZ"],
		"FilterBoxes:=", [0],
		[
			"NAME:PlotOnSurfaceSettings",
			"Filled:=", False,
			"IsoValType:=", "Tone",
			"AddGrid:=", False,
			"MapTransparency:=", True,
			"Refinement:=", 0,
			"Transparency:=", 0,
			"SmoothingLevel:=", 0,
			"ShadingType:=", 0,
			[
				"NAME:Arrow3DSpacingSettings",
				"ArrowUniform:=", True,
				"ArrowSpacing:=", 0,
				"MinArrowSpacing:=", 0,
				"MaxArrowSpacing:=", 0
			],
			"GridColor:=", [255,255,255]
		],
		"EnableGaussianSmoothing:=", False,
		"SurfaceOnly:=", False
	], "Field")

oModule.SetPlotFolderSettings("B",
	[
		"NAME:FieldsPlotSettings",
		"Real time mode:=", True,
		[
			"NAME:ColorMapSettings",
			"ColorMapType:=", "Spectrum",
			"SpectrumType:=", "Rainbow",
			"UniformColor:=", [127,255,255],
			"RampColor:=", [255,127,127]
		],
		[
			"NAME:Scale3DSettings",
			"unit:=", 104,
			"m_nLevels:=", 10,
			"minvalue:=", 2.8E-08,
			"maxvalue:=", 0.0003,
			"log:=", False,
			"IntrinsicMin:=", 2.77330708261318E-08,
			"IntrinsicMax:=", 0.00467342815475659,
			"LimitFieldValuePrecision:=", False,
			"FieldValuePrecisionDigits:=", 4,
			"dB:=", False,
			"AnimationStaticScale:=", False,
			"ScaleType:=", 1,
			"UserSpecifyValues:=", [11,2.80000008642673E-08,0.000467368041992187,0.000934708068847656,0.00140204797363281,0.00186938806152344,0.00233672827148438,0.00280406811523437,0.003271408203125,0.00373874829101562,0.004206087890625,0.00467342822265625],
			"ValueNumberFormatTypeAuto:=", 0,
			"ValueNumberFormatTypeScientific:=", False,
			"ValueNumberFormatWidth:=", 8,
			"ValueNumberFormatPrecision:=", 3
		],
		[
			"NAME:Marker3DSettings",
			"MarkerType:=", 0,
			"MarkerMapSize:=", False,
			"MarkerMapColor:=", False,
			"MarkerSize:=", 0.25
		],
		[
			"NAME:Arrow3DSettings",
			"ArrowType:=", 1,
			"ArrowMapSize:=", False,
			"ArrowMapColor:=", True,
			"ShowArrowTail:=", True,
			"ArrowSize:=", 0.100000001490116,
			"ArrowMinMagnitude:=", -0.499999972266929,
			"ArrowMaxMagnitude:=", 0.504673428154757,
			"ArrowMagnitudeThreshold:=", 0,
			"ArrowMagnitudeFilteringFlag:=", False,
			"ArrowMinIntrinsicMagnitude:=", 0,
			"ArrowMaxIntrinsicMagnitude:=", 1
		],
		[
			"NAME:DeformScaleSettings",
			"ShowDeformation:=", True,
			"MinScaleFactor:=", 0,
			"MaxScaleFactor:=", 1,
			"DeformationScale:=", 0,
			"ShowDeformationOutline:=", False
		]
	])

In [ ]:
## Save project ##
M3D.save_project()